# Tendências das temperaturas máximas diárias

São analisados os 30 anos de 1995 a 2024. Para cada variável, calcula-se a máxima diária e ajusta-se uma regressão linear aos valores diários válidos. As médias anuais são apresentadas apenas para facilitar a visualização da tendência.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

INICIO = "1995-01-01"
FIM = "2024-12-31"

def calcular_maximas_diarias(caminho, nome):
    """Lê uma série horária e devolve suas máximas diárias no período analisado."""
    dados = pd.read_csv(caminho, parse_dates=["data_hora"])
    dados = dados.drop_duplicates(subset="data_hora", keep="first")
    dados["valor"] = dados["valor"].mask(dados["valor"].eq(-99))

    maximas = (
        dados.set_index("data_hora")["valor"]
        .resample("D")
        .max()
        .loc[INICIO:FIM]
        .rename(nome)
    )
    return maximas.dropna()

temperatura_maxima_diaria = calcular_maximas_diarias(
    Path("series_temporais_csv/temperatura_ar_1995_2024.csv"),
    "temperatura_maxima_diaria",
)
bulbo_umido_maximo_diario = calcular_maximas_diarias(
    Path("series_temporais_csv/temperatura_bulbo_umido_1995_2024.csv"),
    "bulbo_umido_maximo_diario",
)

def tendencia_linear(serie):
    """Ajusta y = intercepto + inclinacao * anos_desde_o_inicio."""
    anos_desde_inicio = (serie.index - serie.index.min()).days.to_numpy() / 365.25
    valores = serie.to_numpy(dtype=float)
    inclinacao, intercepto = np.polyfit(anos_desde_inicio, valores, deg=1)
    estimados = intercepto + inclinacao * anos_desde_inicio
    r_quadrado = 1 - (
        np.sum((valores - estimados) ** 2) / np.sum((valores - valores.mean()) ** 2)
    )
    return {
        "inclinacao_c_por_ano": inclinacao,
        "inclinacao_c_por_decada": inclinacao * 10,
        "intercepto": intercepto,
        "r_quadrado": r_quadrado,
    }

tendencia_temperatura = tendencia_linear(temperatura_maxima_diaria)
tendencia_bulbo_umido = tendencia_linear(bulbo_umido_maximo_diario)

resumo_tendencias = pd.DataFrame(
    [
        {
            "variavel": "Temperatura máxima diária",
            "dias_validos": len(temperatura_maxima_diaria),
            "tendencia_°C_por_ano": tendencia_temperatura["inclinacao_c_por_ano"],
            "tendencia_°C_por_decada": tendencia_temperatura["inclinacao_c_por_decada"],
            "R²": tendencia_temperatura["r_quadrado"],
        },
        {
            "variavel": "Bulbo úmido máximo diário",
            "dias_validos": len(bulbo_umido_maximo_diario),
            "tendencia_°C_por_ano": tendencia_bulbo_umido["inclinacao_c_por_ano"],
            "tendencia_°C_por_decada": tendencia_bulbo_umido["inclinacao_c_por_decada"],
            "R²": tendencia_bulbo_umido["r_quadrado"],
        },
    ]
).round(4)

print("Tendências lineares estimadas a partir das máximas diárias (1995–2024):")
print(resumo_tendencias.to_string(index=False))

def plotar_tendencia(eixo, serie, tendencia, titulo, cor):
    medias_anuais = serie.groupby(serie.index.year).mean()
    anos = medias_anuais.index.to_numpy()
    datas_referencia = pd.to_datetime([f"{ano}-07-01" for ano in anos])
    anos_desde_inicio = (datas_referencia - serie.index.min()).days.to_numpy() / 365.25
    linha_tendencia = (
        tendencia["intercepto"] + tendencia["inclinacao_c_por_ano"] * anos_desde_inicio
    )

    eixo.plot(anos, medias_anuais, "o-", color=cor, label="Média anual das máximas diárias")
    eixo.plot(anos, linha_tendencia, "--", color="black", label="Tendência linear diária")
    eixo.set_title(titulo)
    eixo.set_ylabel("Temperatura (°C)")
    eixo.grid(alpha=0.3)
    eixo.legend(loc="best")
    eixo.text(
        0.01, 0.05,
        f"Tendência: {tendencia['inclinacao_c_por_decada']:+.2f} °C/década\nR²: {tendencia['r_quadrado']:.3f}",
        transform=eixo.transAxes,
        va="bottom",
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8},
    )

figura, eixos = plt.subplots(2, 1, figsize=(13, 9), sharex=True, constrained_layout=True)
plotar_tendencia(
    eixos[0], temperatura_maxima_diaria, tendencia_temperatura,
    "Temperatura máxima diária", "tab:red",
)
plotar_tendencia(
    eixos[1], bulbo_umido_maximo_diario, tendencia_bulbo_umido,
    "Temperatura máxima diária de bulbo úmido", "tab:blue",
)
eixos[1].set_xlabel("Ano")
plt.show()